<a href="https://colab.research.google.com/github/srinivasulu-2026/my-first-repo/blob/main/ML_Feature_Engineering_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
rng = np.random.default_rng(11)

N = 5000
SNAPSHOT_DATE = datetime(2025, 1, 1)  # "today" when this data was pulled

cities = ["Bengaluru","Mumbai","Delhi","Hyderabad","Chennai","Pune","Kolkata","Ahmedabad"]
city_w = [0.20,0.18,0.16,0.13,0.10,0.10,0.07,0.06]
city_effect = {"Bengaluru":0.15,"Mumbai":0.05,"Delhi":0.0,"Hyderabad":0.05,
               "Chennai":-0.05,"Pune":0.10,"Kolkata":-0.35,"Ahmedabad":-0.15}

plan_types = ["Monthly","Quarterly","Annual"]
plan_w = [0.50,0.30,0.20]
plan_effect = {"Monthly":-0.35,"Quarterly":0.05,"Annual":0.55}
plan_period_days = {"Monthly":30,"Quarterly":90,"Annual":365}

payment_methods = ["UPI","Card","Wallet","COD"]
pay_w = [0.55,0.25,0.12,0.08]

rows = []
for i in range(N):
    customer_id = f"MEM{10000+i}"
    city = rng.choice(cities, p=city_w)
    plan = rng.choice(plan_types, p=plan_w)
    period_days = plan_period_days[plan]

    days_before_signup = int(rng.integers(60, 900))
    signup_date = SNAPSHOT_DATE - timedelta(days=days_before_signup)

    convert_gap = int(np.clip(rng.exponential(35), 0, min(180, days_before_signup - 1)))
    first_membership_date = signup_date + timedelta(days=convert_gap)

    current_period_start = max(first_membership_date, SNAPSHOT_DATE - timedelta(days=period_days))

    age = int(np.clip(rng.normal(32, 9), 18, 65))
    if rng.random() < 0.08:
        age = np.nan

    orders_last_period = int(np.clip(rng.poisson(9), 0, 60))

    mrp_value = float(np.clip(rng.normal(3200, 1400), 200, 15000))
    discount_pct_true = float(np.clip(rng.beta(2, 6), 0, 0.6))
    amount_paid = round(mrp_value * (1 - discount_pct_true), 2)
    if rng.random() < 0.05:
        amount_paid = np.nan

    support_tickets = int(np.clip(rng.poisson(0.7), 0, 10))

    payment_method = rng.choice(payment_methods, p=pay_w)
    if rng.random() < 0.03:
        payment_method = None

    app_rating = float(np.round(np.clip(rng.normal(4.1, 0.7), 1.0, 5.0), 1))
    rating_missing = rng.random() < 0.22
    if rating_missing:
        app_rating = np.nan

    days_since_last_order = int(np.clip(rng.exponential(18), 0, 200))
    last_order_date = SNAPSHOT_DATE - timedelta(days=days_since_last_order)
    if last_order_date < current_period_start:
        last_order_date = current_period_start
        days_since_last_order = (SNAPSHOT_DATE - last_order_date).days

    logit = 0.75
    logit += plan_effect[plan]
    logit += city_effect[city]
    logit += 0.010 * convert_gap
    logit += 2.0 * discount_pct_true
    logit -= 0.045 * days_since_last_order
    logit -= 0.30 * support_tickets
    logit -= 0.55 if rating_missing else 0
    logit += 0.015 * (orders_last_period - 9)
    logit += rng.normal(0, 0.55)

    prob_renew = 1 / (1 + np.exp(-logit))
    renewed = int(rng.random() < prob_renew)

    rows.append({
        "customer_id": customer_id, "signup_date": signup_date.strftime("%Y-%m-%d"),
        "membership_start_date": first_membership_date.strftime("%Y-%m-%d"),
        "plan_type": plan, "city": city, "age": age,
        "orders_last_period": orders_last_period,
        "mrp_value_last_period": round(mrp_value, 2),
        "amount_paid_last_period": amount_paid, "support_tickets": support_tickets,
        "payment_method": payment_method, "app_rating": app_rating,
        "last_order_date": last_order_date.strftime("%Y-%m-%d"), "renewed": renewed,
    })

df = pd.DataFrame(rows)
df.to_csv("membership_churn.csv", index=False)
print(f"✅ membership_churn.csv — {len(df):,} rows, {df.shape[1]} columns")

✅ membership_churn.csv — 5,000 rows, 14 columns


In [ ]:
df=pd.read_csv("membership_churn.csv",parse_dates=["signup_date", "membership_start_date", "last_order_date"])
SNAPSHOT_DATE = datetime(2025, 1, 1)  # "today" when this data was pulled
print("Shape:",df.shape)
df.head()

Shape: (5000, 14)


,customer_id,signup_date,membership_start_date,plan_type,city,age,orders_last_period,mrp_value_last_period,amount_paid_last_period,support_tickets,payment_method,app_rating,last_order_date,renewed
0,MEM10000,2023-06-26,2023-06-27,Monthly,Bengaluru,29.0,8,3848.35,2466.95,0,Card,2.8,2024-12-17,0
1,MEM10001,2023-06-16,2023-07-24,Monthly,Bengaluru,25.0,6,3546.56,1934.95,0,UPI,5.0,2024-12-31,1
2,MEM10002,2022-09-11,2022-09-19,Monthly,Hyderabad,NaN,8,2469.43,1482.21,0,Wallet,4.4,2024-12-20,1
3,MEM10003,2024-02-10,2024-06-10,Monthly,Delhi,NaN,12,3742.16,2660.25,1,UPI,4.5,2024-12-21,1
4,MEM10004,2024-01-01,2024-01-04,Quarterly,Bengaluru,37.0,8,1170.07,831.22,2,UPI,3.8,2024-11-03,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customer_id              5000 non-null   object        
 1   signup_date              5000 non-null   datetime64[ns]
 2   membership_start_date    5000 non-null   datetime64[ns]
 3   plan_type                5000 non-null   object        
 4   city                     5000 non-null   object        
 5   age                      4588 non-null   float64       
 6   orders_last_period       5000 non-null   int64         
 7   mrp_value_last_period    5000 non-null   float64       
 8   amount_paid_last_period  4731 non-null   float64       
 9   support_tickets          5000 non-null   int64         
 10  payment_method           4847 non-null   object        
 11  app_rating               3876 non-null   float64       
 12  last_order_date          5000 non-

In [ ]:
print("Renewal rate:",df['renewed'].mean().round(3))
#print("Missing values:",df.isna().sum())

Renewal rate: 0.614


In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
ohe=OneHotEncoder(sparse_output=False)

city_encoded = ohe.fit_transform(df[["city"]])

city_encoded_df = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out(["city"]))
print("Original:", df["city"].iloc[0])
print()
city_encoded_df.head(3)

Original: Bengaluru



,city_Ahmedabad,city_Bengaluru,city_Chennai,city_Delhi,city_Hyderabad,city_Kolkata,city_Mumbai,city_Pune
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
ordinal = OrdinalEncoder(categories=[["Monthly", "Quarterly", "Annual"]])
plan_encoded = ordinal.fit_transform(df[["plan_type"]])

comparison = df[["plan_type"]].copy()
comparison["plan_encoded"] = plan_encoded
comparison.drop_duplicates().sort_values("plan_encoded")

,plan_type,plan_encoded
0,Monthly,0.0
4,Quarterly,1.0
13,Annual,2.0


In [ ]:
df.describe()

,signup_date,membership_start_date,age,orders_last_period,mrp_value_last_period,amount_paid_last_period,support_tickets,app_rating,last_order_date,renewed
count,5000,5000,4588.000000,5000.000000,5000.000000,4731.000000,5000.000000,3876.000000,5000,5000.000000
mean,2023-09-14 01:00:46.080000,2023-10-18 19:33:18.720000256,32.010898,9.098800,3219.074122,2411.981300,0.712400,4.050232,2024-12-16 08:11:54.240000,0.614400
min,2022-07-17 00:00:00,2022-07-17 00:00:00,18.000000,1.000000,200.000000,80.000000,0.000000,1.900000,2024-08-21 00:00:00,0.000000
25%,2023-02-07 00:00:00,2023-03-17 00:00:00,25.000000,7.000000,2252.860000,1577.330000,0.000000,3.600000,2024-12-07 00:00:00,0.000000
50%,2023-09-23 00:00:00,2023-10-27 00:00:00,32.000000,9.000000,3212.635000,2328.600000,1.000000,4.100000,2024-12-20 00:00:00,1.000000
75%,2024-04-15 00:00:00,2024-05-22 00:00:00,38.000000,11.000000,4163.037500,3167.035000,1.000000,4.600000,2024-12-28 00:00:00,1.000000
max,2024-11-02 00:00:00,2024-12-31 00:00:00,62.000000,23.000000,7840.590000,6933.080000,5.000000,5.000000,2025-01-01 00:00:00,1.000000
std,NaN,NaN,8.668563,2.996604,1375.246905,1144.933519,0.842513,0.642316,NaN,0.486785


In [ ]:
numeric_sample = df[["orders_last_period", "mrp_value_last_period"]]
print("BEFORE scaling:")
print(numeric_sample.describe().loc[["mean","std","min","max"]].round(1))

scaler = StandardScaler()
scaled = scaler.fit_transform(numeric_sample)
scaled_df = pd.DataFrame(scaled, columns=numeric_sample.columns)

print()
print("AFTER scaling (mean ≈ 0, std ≈ 1 for every column):")
print(scaled_df.describe().loc[["mean","std","min","max"]].round(2))


BEFORE scaling:
      orders_last_period  mrp_value_last_period
mean                 9.1                 3219.1
std                  3.0                 1375.2
min                  1.0                  200.0
max                 23.0                 7840.6

AFTER scaling (mean ≈ 0, std ≈ 1 for every column):
      orders_last_period  mrp_value_last_period
mean               -0.00                   0.00
std                 1.00                   1.00
min                -2.70                  -2.20
max                 4.64                   3.36


# 2. Missing Data Imputation - Filling Gaps

In [ ]:
df.isna().sum()[df.isna().sum()>0]

,0
age,412
amount_paid_last_period,269
payment_method,153
app_rating,1124


In [ ]:
from sklearn.impute import SimpleImputer
age_imputer = SimpleImputer(strategy="median")
age_filled = age_imputer.fit_transform(df[["age"]])

print("Median used to fill:", age_imputer.statistics_[0])
print("Missing before:", df['age'].isna().sum(), " Missing after:", pd.isna(age_filled).sum())



Median used to fill: 32.0
Missing before: 412  Missing after: 0


In [ ]:
pay_imputer = SimpleImputer(strategy="most_frequent")
pay_filled = pay_imputer.fit_transform(df[["payment_method"]])

print("Most frequent value used to fill:", pay_imputer.statistics_[0])
print("Missing before:", df['payment_method'].isna().sum())

Most frequent value used to fill: UPI
Missing before: 153


In [ ]:
df["rating_missing"] = df["app_rating"].isna()
comparison = df.groupby("rating_missing")["renewed"].mean().round(3)
print("Renewal rate by whether app_rating is missing:")
print(comparison)
print()
print(f"Gap: {abs(comparison.iloc[0] - comparison.iloc[1]):.3f} — that's a real, meaningful difference.")

Renewal rate by whether app_rating is missing:
rating_missing
False    0.638
True     0.533
Name: renewed, dtype: float64

Gap: 0.105 — that's a real, meaningful difference.


In [ ]:
rating_imputer = SimpleImputer(strategy="median")
df["app_rating_filled"] = rating_imputer.fit_transform(df[["app_rating"]])

df[["app_rating", "rating_missing", "app_rating_filled"]].head(8)

,app_rating,rating_missing,app_rating_filled
0,2.8,False,2.8
1,5.0,False,5.0
2,4.4,False,4.4
3,4.5,False,4.5
4,3.8,False,3.8
5,4.3,False,4.3
6,4.3,False,4.3
7,NaN,True,4.1


# 3. Feature Creation - Engineering the Features

In [ ]:
df["tenure_days"]=(df['membership_start_date']-df['signup_date']).dt.days
print(df[["signup_date", "membership_start_date", "tenure_days"]].head())
print()
print("Renewal rate by tenure (quartiles):")
print(df.groupby(pd.qcut(df["tenure_days"], 4))["renewed"].mean().round(3))

  signup_date membership_start_date  tenure_days
0  2023-06-26            2023-06-27            1
1  2023-06-16            2023-07-24           38
2  2022-09-11            2022-09-19            8
3  2024-02-10            2024-06-10          121
4  2024-01-01            2024-01-04            3

Renewal rate by tenure (quartiles):
tenure_days
(-0.001, 10.0]    0.540
(10.0, 24.0]      0.586
(24.0, 49.0]      0.623
(49.0, 180.0]     0.714
Name: renewed, dtype: float64


In [ ]:
df["days_since_last_order"] = (SNAPSHOT_DATE - df["last_order_date"]).dt.days
print("Renewal rate by recency (quartiles):")
print(df.groupby(pd.qcut(df["days_since_last_order"], 4))["renewed"].mean().round(3))

Renewal rate by recency (quartiles):
days_since_last_order
(-0.001, 4.0]    0.704
(4.0, 12.0]      0.702
(12.0, 25.0]     0.587
(25.0, 133.0]    0.447
Name: renewed, dtype: float64


Feature engineering is creative work — the best features often come from business intuition, not statistics. AI is a genuinely good brainstorming partner for this, *if* you give it the business context, not just column names.

### The prompt framework — 3 parts

1. **The business problem** — what are we predicting, and why does it matter?
2. **The columns available** — names and what they mean
3. **Ask for feature ideas AND the reasoning** — not just formulas, the "why"

I'm predicting membership churn for a quick-commerce subscription service
(like a paid delivery pass). I want to predict whether a member renews
(1) or churns (0) at the end of their billing period.

Available raw columns:
- signup_date, membership_start_date, last_order_date
- plan_type (Monthly/Quarterly/Annual), city
- age, orders_last_period, mrp_value_last_period, amount_paid_last_period
- support_tickets, payment_method, app_rating

Suggest 5 engineered features I haven't already computed (tenure_days,
days_since_last_order are already done) that
might predict churn, and explain the business reasoning for each.

In [ ]:
df["discount_pct"] = 1 - (df["amount_paid_last_period"] / df["mrp_value_last_period"])

print(df[["mrp_value_last_period","amount_paid_last_period","discount_pct"]].head())
print()
print("Renewal rate by discount used (quartiles):")
print(df.dropna(subset=["discount_pct"]).groupby(
    pd.qcut(df.dropna(subset=["discount_pct"])["discount_pct"], 4))["renewed"].mean().round(3))

   mrp_value_last_period  amount_paid_last_period  discount_pct
0                3848.35                  2466.95      0.358959
1                3546.56                  1934.95      0.454415
2                2469.43                  1482.21      0.399776
3                3742.16                  2660.25      0.289114
4                1170.07                   831.22      0.289598

Renewal rate by discount used (quartiles):
discount_pct
(0.00332, 0.139]    0.548
(0.139, 0.231]      0.602
(0.231, 0.344]      0.631
(0.344, 0.6]        0.677
Name: renewed, dtype: float64


In [ ]:
df["signup_month"] = df["signup_date"].dt.month
df["signup_is_weekend"] = df["signup_date"].dt.dayofweek.isin([5, 6]).astype(int)

print(df[["signup_date", "signup_month", "signup_is_weekend"]].head())
print()
print("Renewal rate by signup month:")
print(df.groupby("signup_month")["renewed"].mean().round(3))

  signup_date  signup_month  signup_is_weekend
0  2023-06-26             6                  0
1  2023-06-16             6                  0
2  2022-09-11             9                  1
3  2024-02-10             2                  1
4  2024-01-01             1                  0

Renewal rate by signup month:
signup_month
1     0.591
2     0.666
3     0.553
4     0.633
5     0.603
6     0.606
7     0.622
8     0.633
9     0.592
10    0.608
11    0.638
12    0.638
Name: renewed, dtype: float64


In [ ]:
# Space to jot down AI suggestions and try implementing one
# Example of the kind of feature AI often suggests: "orders per day of membership"
# (engagement intensity, normalized by how long they've been a member)

df["orders_per_membership_day"] = df["orders_last_period"] / df["tenure_days"].clip(lower=1)

print(df[["orders_last_period", "tenure_days", "orders_per_membership_day"]].head())
print()
print("Renewal rate by orders_per_membership_day (quartiles):")
print(df.groupby(pd.qcut(df["orders_per_membership_day"], 4, duplicates="drop"))["renewed"].mean().round(3))

   orders_last_period  tenure_days  orders_per_membership_day
0                   8            1                   8.000000
1                   6           38                   0.157895
2                   8            8                   1.000000
3                  12          121                   0.099174
4                   8            3                   2.666667

Renewal rate by orders_per_membership_day (quartiles):
orders_per_membership_day
(0.0127, 0.174]    0.713
(0.174, 0.36]      0.608
(0.36, 1.0]        0.587
(1.0, 18.0]        0.546
Name: renewed, dtype: float64


# The basic split

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ["age", "orders_last_period", "mrp_value_last_period",
                 "amount_paid_last_period", "support_tickets", "tenure_days",
                 "days_since_last_order", "discount_pct", "signup_month"]

X = df[feature_cols]
y = df["renewed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} rows   Test: {len(X_test)} rows")
print(f"Train renewal rate: {y_train.mean():.3f}   Test renewal rate: {y_test.mean():.3f}")


Train: 3750 rows   Test: 1250 rows
Train renewal rate: 0.614   Test renewal rate: 0.614


In [ ]:
# ── DEMO: leakage in action ──
# Compute the median age using the FULL dataset vs. using ONLY the training data

full_median = df["age"].median()
train_median = X_train["age"].median()

print(f"Median age (full dataset):  {full_median}")
print(f"Median age (train only):    {train_median}")
print()
print("Small difference here — but imagine an imputer, a scaler, and an encoder")
print("ALL fit on the full dataset. The test set's statistics have quietly leaked")
print("into training. Your evaluation gets optimistic, and production performance")
print("disappoints because the real world doesn't leak test-set information.")


Median age (full dataset):  32.0
Median age (train only):    31.0

Small difference here — but imagine an imputer, a scaler, and an encoder
ALL fit on the full dataset. The test set's statistics have quietly leaked
into training. Your evaluation gets optimistic, and production performance
disappoints because the real world doesn't leak test-set information.
